In [19]:
import os

os.environ["HF_HOME"] = "D:/huggingface_cache"

# ── Paths ─────────────────────────────────────────────────────────────────────
RESUMES_DIR   = "../data/raw/resumes"
IMAGES_DIR    = "../data/raw/page_images"
DOCTAGS_FILE  = "../data/extracted_text/doctags_progress.txt"

os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DOCTAGS_FILE), exist_ok=True)

# ── Global tuning knobs ───────────────────────────────────────────────────────
PDF_DPI        = 100  
BATCH_SIZE     = 4
MAX_NEW_TOKENS = 8192

print("Paths configured.")
print(f"DPI={PDF_DPI} | BATCH_SIZE={BATCH_SIZE} | MAX_NEW_TOKENS={MAX_NEW_TOKENS}")

Paths configured.
DPI=100 | BATCH_SIZE=4 | MAX_NEW_TOKENS=8192


In [5]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_NAME = "ibm-granite/granite-docling-258M"
device     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

granite_doc_processor = AutoProcessor.from_pretrained(MODEL_NAME)

granite_doc_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype             = torch.bfloat16,
    attn_implementation = "sdpa",
    low_cpu_mem_usage = True,
).to(device)

granite_doc_model.eval()
print(f"Granite Docling loaded on {device}")

Using device: cuda
Granite Docling loaded on cuda


In [16]:
import json
import pymupdf
from PIL import Image
from pathlib import Path

def pdf_to_images(
    resumes_dir   : str,
    output_dir    : str,
    dpi           : int  = PDF_DPI,
    skip_existing : bool = True,
):
    """
    Renders each PDF page to PNG at the specified DPI.
    DPI is the only knob — lower DPI = smaller pixels = faster vision encoder.
    No resizing or cropping ever happens; full page content is always preserved.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    meta_path  = output_dir / "metadata.json"

    if skip_existing and meta_path.exists():
        with open(meta_path, encoding="utf-8") as f:
            existing_meta = json.load(f)
        existing_images = {e["image_path"] for e in existing_meta}
        print(f"Resuming: {len(existing_meta)} pages already in metadata")
    else:
        existing_meta   = []
        existing_images = set()

    resumes_dir = Path(resumes_dir)
    pdf_files   = sorted(resumes_dir.rglob("*.pdf")) + sorted(resumes_dir.rglob("*.PDF"))

    if not pdf_files:
        print(f"No PDFs found under {resumes_dir}")
        return [], []

    print(f"Found {len(pdf_files)} PDFs across all category folders")

    new_meta = []

    for pdf_path in pdf_files:
        category = pdf_path.parent.name
        pdf_stem = pdf_path.stem

        try:
            doc = pymupdf.open(str(pdf_path))
        except Exception as e:
            print(f"  SKIP (cannot open): {pdf_path.name} — {e}")
            continue

        num_pages = len(doc)

        for page_num, page in enumerate(doc):
            safe_cat  = category.replace(" ", "_").replace("/", "-")
            safe_stem = pdf_stem.replace(" ", "_")[:60]
            img_name  = f"{safe_cat}__{safe_stem}__p{page_num:04d}.png"
            img_path  = str(output_dir / img_name)

            if skip_existing and img_path in existing_images:
                continue

            # Render at DPI — this is the only size control, no post-processing
            pix   = page.get_pixmap(dpi=dpi)
            image = pix.pil_image().convert("RGB")
            del pix

            image.save(img_path)
            del image

            new_meta.append({
                "image_path"  : img_path,
                "pdf_path"    : str(pdf_path),
                "pdf_name"    : pdf_path.name,
                "pdf_stem"    : pdf_stem,
                "category"    : category,
                "page_number" : page_num,
                "total_pages" : num_pages,
            })

        doc.close()
        if num_pages > 0:
            print(f"  {category}/{pdf_path.name}: {num_pages} pages")

    all_meta = existing_meta + new_meta
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(all_meta, f, indent=2)

    image_paths = [e["image_path"] for e in all_meta]
    print(f"\nTotal pages : {len(image_paths)} ({len(new_meta)} new this run)")
    print(f"Metadata    : {meta_path}")
    return image_paths, all_meta

image_paths, metadata = pdf_to_images(RESUMES_DIR, IMAGES_DIR)
print(f"{len(image_paths)} total page images ready")

Found 5172 PDFs across all category folders
  Accountant resumes/Image_10.pdf: 1 pages
  Accountant resumes/Image_11.pdf: 1 pages
  Accountant resumes/Image_12.pdf: 1 pages
  Accountant resumes/Image_14.pdf: 1 pages
  Accountant resumes/Image_16.pdf: 1 pages
  Accountant resumes/Image_17.pdf: 1 pages
  Accountant resumes/Image_18.pdf: 1 pages
  Accountant resumes/Image_19.pdf: 1 pages
  Accountant resumes/Image_2.pdf: 1 pages
  Accountant resumes/Image_20.pdf: 1 pages
  Accountant resumes/Image_21.pdf: 1 pages
  Accountant resumes/Image_22.pdf: 1 pages
  Accountant resumes/Image_24.pdf: 1 pages
  Accountant resumes/Image_25.pdf: 1 pages
  Accountant resumes/Image_26.pdf: 1 pages
  Accountant resumes/Image_29.pdf: 1 pages
  Accountant resumes/Image_30.pdf: 1 pages
  Accountant resumes/Image_31.pdf: 1 pages
  Accountant resumes/Image_32.pdf: 1 pages
  Accountant resumes/Image_33.pdf: 1 pages
  Accountant resumes/Image_34.pdf: 1 pages
  Accountant resumes/Image_39.pdf: 1 pages
  Accountan

In [17]:
from collections import Counter

print(f"Total pages: {len(image_paths)}")
print(f"\nPages per category:")
cat_counts = Counter(e["category"] for e in metadata)
for cat, count in sorted(cat_counts.items()):
    print(f"  {cat:45s} {count:>4} pages")

print(f"\nSample metadata entries:")
for e in metadata[:3]:
    print(f"  {e}")

# ── Prompt template ───────────────────────────────────────────────────────────
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."},
        ],
    },
]

prompt = granite_doc_processor.apply_chat_template(
    conversation          = messages,
    add_generation_prompt = True,
)
print("\nPrompt template ready.")
print(f"Prompt length: {len(prompt)} chars")

Total pages: 5172

Pages per category:
  Accountant resumes                             134 pages
  Advocate resumes                               188 pages
  Agricultural resumes                           136 pages
  Apparel resumes                                102 pages
  Architects resumes                             120 pages
  Arts resumes                                   136 pages
  Automobile resumes                              80 pages
  Aviation resumes                               102 pages
  BPO resumes                                     58 pages
  Banking resumes                                 82 pages
  Blockchain resumes                              16 pages
  Building _Construction resumes                 110 pages
  Business Analyst resumes                       112 pages
  Civil Engineer resumes                         146 pages
  Consultant resumes                             142 pages
  Database resumes                               126 pages
  Designing resum

In [20]:
import os
import re
import time
import threading
import torch
from PIL import Image

# ── RESUME FROM DISK ──────────────────────────────────────────────────────────
all_doc_tags = []
if os.path.exists(DOCTAGS_FILE):
    with open(DOCTAGS_FILE, "r", encoding="utf-8") as f:
        content = f.read()
    all_doc_tags = [t for t in content.split("\n<<PAGE_BREAK>>\n") if t.strip()]
    print(f"Resumed: {len(all_doc_tags)} pages already processed")
else:
    print("Starting fresh")

start_page  = len(all_doc_tags)
start_total = time.time()

progress_file = open(DOCTAGS_FILE, "a", encoding="utf-8")

# ── PREFETCH ──────────────────────────────────────────────────────────────────
_prefetch_result = None
_prefetch_thread = None

def _load_image(path):
    global _prefetch_result
    _prefetch_result = Image.open(path).convert("RGB")

def prefetch_next(path):
    global _prefetch_thread
    _prefetch_thread = threading.Thread(target=_load_image, args=(path,), daemon=True)
    _prefetch_thread.start()

def get_prefetched():
    if _prefetch_thread is not None:
        _prefetch_thread.join()
    return _prefetch_result

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
for i in range(start_page, len(image_paths)):

    batch_start = time.time()
    path        = image_paths[i]

    if i == start_page:
        image = Image.open(path).convert("RGB")
    else:
        image = get_prefetched()

    if i + 1 < len(image_paths):
        prefetch_next(image_paths[i + 1])

    # ── ENCODE ────────────────────────────────────────────────────────────────
    inputs = granite_doc_processor(
        text           = prompt,
        images         = image,
        return_tensors = "pt",
    ).to(device)

    del image

    # ── GENERATE ──────────────────────────────────────────────────────────────
    with torch.inference_mode():
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            generated_ids = granite_doc_model.generate(
                **inputs,
                max_new_tokens = MAX_NEW_TOKENS,
                do_sample      = False,
                use_cache      = True,
            )

    prompt_length         = inputs.input_ids.shape[1]
    trimmed_generated_ids = generated_ids[:, prompt_length:]

    raw = granite_doc_processor.decode(
        trimmed_generated_ids[0],
        skip_special_tokens=False,
    ).lstrip()

    # ── VALIDATION ────────────────────────────────────────────────────────────
    close_count = raw.count("</doctag>")
    if close_count == 0:
        meta = metadata[i] if i < len(metadata) else {}
        print(f"  WARNING page {i+1} ({meta.get('pdf_name', '')}): truncated")
        tag = raw
    elif close_count == 1:
        tag = raw
    else:
        print(f"  WARNING page {i+1}: {close_count} pages merged — splitting")
        individual = re.findall(r"<doctag>.*?</doctag>", raw, re.DOTALL)
        if individual:
            tag = individual[0]
            print(f"    → Kept first of {len(individual)} splits")
        else:
            tag = raw

    all_doc_tags.append(tag)
    progress_file.write(tag + "\n<<PAGE_BREAK>>\n")
    progress_file.flush()

    del inputs, generated_ids, trimmed_generated_ids
    if i % 5 == 0:
        torch.cuda.empty_cache()

    # ── PROGRESS ──────────────────────────────────────────────────────────────
    elapsed    = time.time() - start_total
    batch_time = time.time() - batch_start
    meta_entry = metadata[i] if i < len(metadata) else {}
    pages_done = len(all_doc_tags) - start_page
    s_per_page = elapsed / max(pages_done, 1)
    eta_min    = (len(image_paths) - len(all_doc_tags)) * s_per_page / 60

    print(
        f"[{len(all_doc_tags):>5}/{len(image_paths)}] "
        f"{meta_entry.get('category', '')[:28]:28s} | "
        f"{batch_time:.1f}s/page | "
        f"ETA {eta_min:.0f}min"
    )

progress_file.close()

total_elapsed = time.time() - start_total
pages_done    = len(all_doc_tags) - start_page
print(
    f"\nDone. {len(all_doc_tags)} pages total | "
    f"{total_elapsed/60:.1f}min | "
    f"{total_elapsed / max(pages_done, 1):.1f}s/page avg"
)

Resumed: 38 pages already processed
[   39/5172] Accountant resumes           | 42.0s/page | ETA 3597min
[   40/5172] Accountant resumes           | 90.1s/page | ETA 5650min
[   41/5172] Accountant resumes           | 54.4s/page | ETA 5316min
[   42/5172] Accountant resumes           | 62.7s/page | ETA 5327min
[   43/5172] Accountant resumes           | 44.7s/page | ETA 5026min
  WARNING page 44 (Image_67.pdf): truncated
[   44/5172] Accountant resumes           | 372.9s/page | ETA 9498min
[   45/5172] Accountant resumes           | 71.3s/page | ETA 9011min
[   46/5172] Accountant resumes           | 65.4s/page | ETA 8581min
[   47/5172] Accountant resumes           | 57.5s/page | ETA 8172min
[   48/5172] Accountant resumes           | 125.0s/page | ETA 8421min
[   49/5172] Accountant resumes           | 180.0s/page | ETA 9051min
[   50/5172] Accountant resumes           | 83.3s/page | ETA 8887min
[   51/5172] Accountant resumes           | 57.5s/page | ETA 8580min
[   52/5172] Account

KeyboardInterrupt: 

In [13]:
# Check the slow pages — what do they look like?
slow_pages = [15, 16, 17, 18, 19, 20]  # 0-indexed, adjust to your actual slow ones
for idx in slow_pages:
    if idx < len(metadata):
        m = metadata[idx]
        print(f"Page {idx+1}: {m['pdf_name']} | category: {m['category']} | page {m['page_number']+1}/{m['total_pages']}")
        # Check image size
        from PIL import Image
        img = Image.open(m['image_path'])
        print(f"         Image size: {img.size} | mode: {img.mode}")
        img.close()

Page 16: Image_29.pdf | category: Accountant resumes | page 1/1
         Image size: (1224, 1584) | mode: RGB
Page 17: Image_30.pdf | category: Accountant resumes | page 1/1
         Image size: (1800, 2547) | mode: RGB
Page 18: Image_31.pdf | category: Accountant resumes | page 1/1
         Image size: (2480, 3509) | mode: RGB
Page 19: Image_32.pdf | category: Accountant resumes | page 1/1
         Image size: (1190, 1683) | mode: RGB
Page 20: Image_33.pdf | category: Accountant resumes | page 1/1
         Image size: (1200, 1698) | mode: RGB
Page 21: Image_34.pdf | category: Accountant resumes | page 1/1
         Image size: (3717, 5261) | mode: RGB


In [ ]:
from collections import defaultdict

with open(DOCTAGS_FILE, "r", encoding="utf-8") as f:
    content = f.read()

pages = [p for p in content.split("\n<<PAGE_BREAK>>\n") if p.strip()]
print(f"Pages in file : {len(pages)}")
print(f"Pages in RAM  : {len(all_doc_tags)}")
print(f"Pages expected: {len(image_paths)}")

# ── Truncation check with per-category breakdown ──────────────────────────────
truncated_indices = [i for i, p in enumerate(pages) if "</doctag>" not in p]

if truncated_indices:
    print(f"\nWARNING: {len(truncated_indices)} truncated pages")
    cat_truncations = defaultdict(int)
    for idx in truncated_indices:
        if idx < len(metadata):
            cat_truncations[metadata[idx]["category"]] += 1
    print("Truncations per category:")
    for cat, count in sorted(cat_truncations.items(), key=lambda x: -x[1]):
        print(f"  {cat:45s} {count:>4} truncated")
    print("  → Increase MAX_NEW_TOKENS for affected categories only and re-run")
else:
    print("\nAll pages have complete </doctag> closing tags. ✓")

# ── Preview ───────────────────────────────────────────────────────────────────
print(f"\nFirst page preview (first 500 chars):")
print(pages[0][:500] if pages else "(empty)")

In [ ]:
import gc

try:
    del granite_doc_model
    del granite_doc_processor
    image_store.clear()       # free the potentially multi-GB RAM image cache
    torch.cuda.empty_cache()
    gc.collect()
    print("Granite unloaded — VRAM + RAM freed")
except NameError:
    torch.cuda.empty_cache()
    gc.collect()
    print("Granite was not in session")